# SPI-TAP non-interactive run
Set parameters, run until source shortlist, then continue.

## Selections and directory build

In [ ]:
import os
import spitap.spi_obs as st

# directory with all analysis results
obsdir = "obs"

RUN = {
    "src_dir": "test",
    "full_name": "Crab", "ra": None, "dec": None, #83.6331, 22.0145,
    # "full_name": "none", "ra": 83.6331, "dec": 22.0145,
    "date_start": "2023-01-01",
    "date_end": "2023-04-01",
    "off_angle": 10.0, # angle from main source to pointing centers
    "evt_type": "SE",
    "binning_type": "log",
    "emin": 20.0,
    "emax": 400.0,
    "nbins": 25,
    "e_channels_bounds": None,
    "e_channels_scales": None,
    "overwrite_date_dir": False,
    "skip_spiselectscw_par": True,
    "zenith_angle": 10.0, # angle from pointing centers to sources 
}

Loading spi_obs module...



In [2]:
# initial_dir = os.getcwd()
initial_dir = "/home/tbouchet/spi-tap" # REPLACE WITH LOCATION OF NOTEBOOK
initial_env = os.environ.copy()

obs = st.ObsSPI(
    main_dir=obsdir,
    initial_dir=initial_dir,
    initial_env=initial_env,
    testrun=False,
)
nearby_df = obs.run_analysis_noninteractive(**RUN, src_sel=None)
nearby_df

Loading config paths...
spi_cat_path = /home/tbouchet/cat/spi_cat_extended_2048_varpars.fits
gnrl_cat_path = /home/tbouchet/cat/nrt_cat_tristan.fits
data_dir = /mnt/kiwi/INTEGRAL/Private_low
scw_db_path = /home/tbouchet/ScwDB_0016-2887_reduced_filterGTI.fits.gz
all_revs_path = /data1/ipp_afs_mirror/integral/shared_analysis/cookbook/revolutions/all_revs.fits
bkg_db_dir = /home/tbouchet/BKG_DB
inRMFfile = /data1/ipp_afs_mirror/integral/data/ic/current/ic/spi/rsp/spi_rmf_grp_0002.fits
cfitsio_templates_dir = /data1/ipp_afs_mirror/integral/software/osa/osa-10.0/linux64_sw-10.0/templates
spi_off_det = /data1/ipp_afs_mirror/integral/software/local/spiselectscw/current/spi_off_det.fits
spi_gnrl_bti = /data1/ipp_afs_mirror/integral/data/ic/spi/lim/spi_gnrl_bti_0005.fits
spiselectscw_cmd = /data1/ipp_afs_mirror/integral/software/local/spiselectscw/4.02/amd64_sles11_g++/spiselectscw
spimodfit_cmd = /data1/ipp_afs_mirror/integral/software/local/spimodfit/3.2/amd64_sles11_g++/spimodfit
rmfgen_cmd 

,NAME,RA_OBJ,DEC_OBJ,SEP_DEG,Flux (mCrab)
0,Crab,83.623657,22.017742,0.009440,1000.000000
1,A0535+32,84.733665,26.313475,4.414605,67.603745
2,4U 0614+091,94.271103,9.140375,16.437109,25.301205


## Run analysis

In [3]:
# choose source selection manually here (0 to select all)
src_sel = 0
# src_sel = [0, 1, 3]

In [5]:
FIT = {
    "main_src_var_unit": "d",
    "main_src_var_n": "3",
    "main_src_var_type": "i",
    # variability of all other sources
    "src_var_n": "0",
    "src_var_unit": "d",
    "src_var_type": "i",
    # variability of background
    "bkg_var_n": "2",
    "bkg_var_unit": "p",
    "bkg_var_type": "i",
    "spimodfit_clobber": True,
    "response_clobber": "y",
}

obs.select_brightest(src_sel)
# this method can be repeated for specific sources by changing src_name
obs.select_src_var(
    main_src_var_unit=FIT["main_src_var_unit"],
    main_src_var_n=FIT["main_src_var_n"],
    main_src_var_type=FIT["main_src_var_type"],
    src_name= None,
)
# change variability of all remaining sources
obs.make_spimodfit_par(
    src_var_n=FIT["src_var_n"],
    src_var_unit=FIT["src_var_unit"],
    src_var_type=FIT["src_var_type"],
    bkg_var_n=FIT["bkg_var_n"],
    bkg_var_unit=FIT["bkg_var_unit"],
    bkg_var_type=FIT["bkg_var_type"],
    spimodfit_clobber=FIT["spimodfit_clobber"],
)
obs.analyze_spimodfit(verbose=True)
obs.generate_response(clobber=FIT["response_clobber"])

Current spimodfit run directory: /home/tbouchet/SPI_SOURCES/obs/test/2023-01-01_2023-04-01_10deg/20_400_25log_SE
FOV catalog saved at /home/tbouchet/SPI_SOURCES/obs/test/2023-01-01_2023-04-01_10deg/20_400_25log_SE/cat/fov_cat_0di_2pi.fits
Creating spimodfit parameter file...
Parameter file created: spimodfit.fit_0di_2pi.par
Launching spimodfit (RUN_ID: fit_0di_2pi)...
Running command.   
Command completed with exit status: 0
Output logged to spimodfit.log
Fit directory at /home/tbouchet/SPI_SOURCES/obs/test/2023-01-01_2023-04-01_10deg/20_400_25log_SE/fit_0di_2pi
spimodfit ran successfully. 5 spectra created.
Pearson's chi2 stat / dof for each energy bin (threshold = 4 sigma)
bin  E range (keV)     chi2 red./dof
0  : 20.0   - 22.5   : 1.16 (544 dof) (thres= ± 0.243)
1  : 22.5   - 25.0   : 0.99 (544 dof) (thres= ± 0.243)
2  : 25.0   - 28.5   : 1.04 (544 dof) (thres= ± 0.243)
3  : 28.5   - 32.0   : 1.08 (544 dof) (thres= ± 0.243)
4  : 32.0   - 36.0   : 1.24 (544 dof) (thres= ± 0.243)
5  :

0

# Reset analysis

In [5]:
os.environ.clear()
os.environ.update(initial_env)
os.chdir(initial_dir)
print("Environment and directory restored.")

Environment and directory restored.


# Check residuals and light-curves (WIP)

In [ ]:
obs

In [ ]:
class SPIResult:
    def __init__(self):
        pass
    